Task 13 — Direct Preference Optimization (DPO)

This task uses PyTorch, TRL, and Hugging Face Accelerate. It trains a model using preference pairs: chosen response and rejected response.

In [1]:
# Import Libraries

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
# Load Model

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
# Preference Data

prompt = "What is machine learning?"

chosen = "Machine learning allows computers to learn from data."

rejected = "Machine learning is only used for gaming."

In [4]:
# Tokenize Chosen Response

chosen_tokens = tokenizer(
    prompt + chosen,
    return_tensors="pt"
)

print(chosen_tokens["input_ids"].shape)

torch.Size([1, 14])


In [5]:
# Tokenize Rejected Response

rejected_tokens = tokenizer(
    prompt + rejected,
    return_tensors="pt"
)

print(rejected_tokens["input_ids"].shape)

torch.Size([1, 13])


In [6]:
# Calculate Chosen Log Probability

chosen_output = model(
    **chosen_tokens
)

chosen_log_prob = torch.log_softmax(
    chosen_output.logits,
    dim=-1
).mean()

print(chosen_log_prob)

tensor(-18.4774, grad_fn=<MeanBackward0>)


In [7]:
# Calculate Rejected Log Probability

rejected_output = model(
    **rejected_tokens
)

rejected_log_prob = torch.log_softmax(
    rejected_output.logits,
    dim=-1
).mean()

print(rejected_log_prob)

tensor(-17.8446, grad_fn=<MeanBackward0>)


In [8]:
# Calculate Preference Loss

loss = -torch.log(
    torch.sigmoid(
        chosen_log_prob - rejected_log_prob
    )
)

print("DPO Loss:", loss.item())

DPO Loss: 1.0587539672851562


In [9]:
# Backward Pass

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5
)

loss.backward()

optimizer.step()

optimizer.zero_grad()

In [10]:
# Results

print("Chosen Log Probability:", chosen_log_prob.item())
print("Rejected Log Probability:", rejected_log_prob.item())
print("DPO Loss:", loss.item())

Chosen Log Probability: -18.477352142333984
Rejected Log Probability: -17.8446044921875
DPO Loss: 1.0587539672851562
